# 08 - RS-PPO gegen ArmoRM: bewusst zirkulaerer Upper-Bound-Test

Dieses Notebook trainiert PPO **bewusst** gegen ArmoRM und evaluiert danach wieder gegen ArmoRM. Das ist **zirkulaer** und kein Proxy-Validitaets-Experiment.

## Was dieses Notebook zeigen kann und was nicht

**Ungueltig durch Zirkularitaet - nicht berichten:**

- Keine RQ2-Proxy-Validitaet: keine Aussage der Form "R ist ein valider Proxy fuer Reward".
- Keine Spearman-Gate-Zahl als Proxy-Validierung. Falls Rho berechnet wird, ist es reine Diagnostik und wird als `circular_do_not_report: true` markiert.
- Keine Generalisierung ueber ArmoRM hinaus.

**Gueltig trotz Zirkularitaet:**

- **Obere Schranke (Hauptbeitrag):** PPO gegen r_i macht delta_j ungefaehr zum Reward-Gradientenschritt -> Assumption 1 gilt fast per Konstruktion -> dies ist das *Best-Case-Regime* fuer f(p,R). Schlaegt f(p,R) hier lambda=p **nicht**, dann nirgends. Ein Negativbefund wird dadurch **staerker**.
- **R-minus:** Bricht PPO die konfliktfreie Geometrie? (rein geometrisch, 0 Reward-Queries)
- **Wall A / R2:** Bleibt U_p(theta(lambda)) auf quality-Achsen nichtlinear? (Eigenschaft von ArmoRM x Interpolation, kein Trainingsartefakt)
- **LMC:** Sind theta_SFT und die PPO-Spezialisten linear verbunden? (RS Working Hyp. 1)

**Kritische Bedingung fuer die obere Schranke:** der Horizont muss **kurz** bleiben. Plateaut die Reward-Kurve, ist PPO auskonvergiert, `delta != eta * grad r`, und das Best-Case-Argument verliert seine Grundlage. Deshalb wird `reward_plateaued` pro Achse geloggt und im Verdict gefuehrt.

Ein bindender Lauf wird vorregistriert. Ergebnis in beide Richtungen akzeptieren; keine Retrain-schauen-Retrain-Schleife.

---

### Audit-Fixes gegenueber dem ersten Entwurf

| # | Befund | Fix |
|---|---|---|
| F1 | `CFG`-Mutationen gingen ueber die `subprocess`-Grenze verloren -> Adapter landeten im falschen Verzeichnis, 4-bit/Batchsize/Steps unwirksam | Training laeuft **in-process** (`rs_ppo.run_ppo(...)`); zusaetzlich CLI-Flags im Skript |
| F2 | ArmoRM-Head-Indizes 0..4 hartkodiert (= QRM-Bugklasse) | Index wird **per Name aus `config.id2label`** aufgeloest und asserted |
| F3 | Batched Reward-Scoring ungeprueft; Attention-Mask ueber `!= pad_id` maskiert echte `<|eot_id|>` weg | Maske aus **echten Laengen**; Padding-Seite **empirisch** aufgeloest; `batched == single` wird bewiesen, sonst Fallback auf `batch_size=1` |
| F4 | `pad_token_id or eos_token_id` -> falsy-Bug bei `pad_token_id == 0` | expliziter `is None`-Check |
| F5 | Head-Sanity lief auf **HelpSteer2-Referenzantworten**, nicht auf der Verteilung, die PPO erzeugt | Sanity laeuft auf **theta_SFT-Generierungen** (16-32 Tokens, identische Gen-Config) |
| F6 | Kein Plateau-Check -> Upper-Bound-Argument unbelegt | `detect_reward_plateau()` pro Achse, im Verdict |
| F7 | `floor_collapse_risk` aus dem Vorzeichen von R-minus statt aus dem LP | Floor-Kollaps direkt aus `max min_i (Rv)_i <= tol` |
| F8 | Reward-Collection / Merge / LMC nicht implementiert (primaerer Endpunkt fehlte) | implementiert: `reward_matrix.npy` ueber B, echtes `lambda*`-Mergen, Bootstrap-CI, LMC-Kurven |
| F9 | Tote Config-Keys (`MAX_NEW_TOKENS`, `REPETITION_PENALTY`, ...) | entfernt bzw. tatsaechlich durchgereicht |


## Setup

Repository, Dependencies, Seeds, Config.


In [ ]:
%cd /content

import os, sys, json, math, random, shutil, subprocess, time, zipfile
from datetime import datetime, timezone
from pathlib import Path

repo_path = Path('/content/master-thesis')
repo_url = 'https://github.com/NZhang137/master-thesis.git'
if (repo_path / '.git').is_dir():
    print('Repository exists; pulling latest changes.')
    subprocess.run(['git', '-C', str(repo_path), 'pull', '--ff-only'], check=False)
else:
    print('Repository missing; cloning from GitHub.')
    if repo_path.exists():
        shutil.rmtree(repo_path)
    subprocess.run(['git', 'clone', repo_url, str(repo_path)], check=True)

%cd /content/master-thesis

!pip install -q "transformers==4.40.0" "peft==0.10.0" "accelerate==0.29.3" "trl==0.8.6" bitsandbytes datasets scipy numpy pandas matplotlib safetensors

import numpy as np
import pandas as pd
import torch

CONFIG = {
    'SEED': 137,
    'OUTPUT_DIR': 'results/rs_ppo_armorm_circular',
    'OUTPUT_ZIP': 'rs_ppo_armorm_circular_outputs.zip',

    'BASE_MODEL': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0',
    'ARMORM_MODEL': 'RLHFlow/ArmoRM-Llama3-8B-v0.1',
    'ATTRIBUTES': ['helpfulness', 'correctness', 'coherence', 'complexity', 'verbosity'],

    # --- circularity is a deliberate, documented decision ---
    'CIRCULAR_ARMORM_ACKNOWLEDGED': True,

    # --- phase switches (expensive stages behind explicit flags) ---
    'RUN_HEAD_SANITY': True,
    'RUN_SFT': False,
    'RUN_PPO': False,
    'RUN_GEOMETRY': False,
    'RUN_REWARD_COLLECTION': False,   # F8: builds reward_matrix.npy over B (expensive)
    'RUN_LMC': False,                 # F8: theta_SFT <-> theta_i interpolation curves
    'RUN_FINAL_MERGE': False,         # F8: primary endpoint, real lambda* merges

    # --- F5: head sanity runs on POLICY GENERATIONS, not dataset responses ---
    'HEAD_SANITY_SAMPLES': 200,
    'HEAD_SANITY_SPLIT': 'validation',
    'HEAD_SANITY_ON_GENERATIONS': True,
    'HEAD_STD_MIN': 1e-6,
    'HEAD_UNIQUE_MIN': 10,

    # --- reward collection over the search set B ---
    'REWARD_NUM_PROMPTS': 80,
    'REWARD_PROMPT_SPLIT': 'validation',
    'REWARD_PROMPT_OFFSET': 160,      # disjoint from the v5 confirmatory slice [80:160]

    # --- PPO (RS Table 1 defaults; NO tuning -- "Training ist Nebensache") ---
    'PPO_AXES': ['helpfulness', 'correctness', 'coherence', 'complexity', 'verbosity'],
    'PPO_BATCH_SIZE': 64,
    'PPO_MINI_BATCH_SIZE': 8,
    'PPO_TOTAL_STEPS': 200,
    'PPO_N_PROMPTS': 2005,
    'ARMORM_REWARD_BATCH_SIZE': 8,
    'ARMORM_LOAD_IN_4BIT': True,

    # --- F6: short-horizon guard ---
    'PLATEAU_WINDOW': 50,
    'PLATEAU_SLOPE_EPS': 0.02,

    # --- selection / evaluation ---
    'M1PLUS_RHO': 0.5,
    'C1PP_C': 0.5,                    # C1++ trust-region radius factor
    'C1PP_EPS': 1e-8,
    'DM_DENOM_MIN': 1e-3,             # guard: Delta m% denominator
    'BOOTSTRAP_N': 2000,
    'BOOTSTRAP_SEED': 137,
    'SEARCH_SET_SEED': 137,
    'SEARCH_SET_DIRICHLET': 64,
    'LMC_GRID': [0.0, 0.25, 0.5, 0.75, 1.0],
    'FLOOR_TOL': 1e-9,                # F7: floor collapse iff LP value <= tol
    'MIN_GPU_MEMORY_GB': 35.0,
}

PROJECT_ROOT = Path.cwd().resolve()
OUTPUT_DIR = (PROJECT_ROOT / CONFIG['OUTPUT_DIR']).resolve()
RS_RUNS_DIR = OUTPUT_DIR / 'rs_runs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RS_RUNS_DIR.mkdir(parents=True, exist_ok=True)

random.seed(CONFIG['SEED']); np.random.seed(CONFIG['SEED']); torch.manual_seed(CONFIG['SEED'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG['SEED'])

assert CONFIG['CIRCULAR_ARMORM_ACKNOWLEDGED'] is True
assert CONFIG['SEED'] == 137
assert CONFIG['REWARD_PROMPT_OFFSET'] >= 160, 'must not overlap the v5 confirmatory slice'

print('Project root:', PROJECT_ROOT)
print('Output dir:  ', OUTPUT_DIR)
print('RS runs dir: ', RS_RUNS_DIR)
print(json.dumps(CONFIG, indent=2, sort_keys=True))


## Import, Firewall-Test und Pre-Registration

**F1:** `train_rs_ppo` wird importiert und **in-process** konfiguriert. Kein `subprocess` mehr - dadurch greifen `out_dir`, `batch_size`, `total_ppo_steps`, `n_prompts` und 4-bit tatsaechlich. (Vorher gingen genau diese fuenf Zuweisungen ueber die Prozessgrenze verloren; die Adapter waeren nach `./rs_runs/` statt nach `OUTPUT_DIR/rs_runs/` geschrieben worden und Zelle "Geometrie" haette sie nicht gefunden.)

Die Firewall muss **ohne** Flag hart fehlschlagen und **mit** Flag eine laute Zirkularitaetswarnung liefern. Beide Pfade werden getestet.


In [ ]:
import importlib
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

rs_ppo = importlib.import_module('scripts.train_rs_ppo')
rs_ppo = importlib.reload(rs_ppo)

# --- F1: in-process overrides. These now actually take effect. ---
rs_ppo.apply_overrides(
    out_dir=str(RS_RUNS_DIR),
    batch_size=CONFIG['PPO_BATCH_SIZE'],
    mini_batch_size=CONFIG['PPO_MINI_BATCH_SIZE'],
    total_ppo_steps=CONFIG['PPO_TOTAL_STEPS'],
    n_prompts=CONFIG['PPO_N_PROMPTS'],
    armorm_model=CONFIG['ARMORM_MODEL'],
    armorm_load_in_4bit=CONFIG['ARMORM_LOAD_IN_4BIT'],
    armorm_reward_batch_size=CONFIG['ARMORM_REWARD_BATCH_SIZE'],
    plateau_window=CONFIG['PLATEAU_WINDOW'],
    plateau_slope_eps=CONFIG['PLATEAU_SLOPE_EPS'],
)
assert rs_ppo.CFG['out_dir'] == str(RS_RUNS_DIR), 'override did not stick'
assert rs_ppo.CFG['total_ppo_steps'] == CONFIG['PPO_TOTAL_STEPS']
print('[cfg] rs_ppo.CFG out_dir =', rs_ppo.CFG['out_dir'])
print('[cfg] rs_ppo.CFG steps   =', rs_ppo.CFG['total_ppo_steps'],
      ' batch =', rs_ppo.CFG['batch_size'], ' 4bit =', rs_ppo.CFG['armorm_load_in_4bit'])

# --- firewall: must BLOCK without the flag ---
blocked = False
try:
    rs_ppo.check_reward_firewall('helpfulness', CONFIG['ARMORM_MODEL'],
                                 circular_armorm_acknowledged=False)
except AssertionError as error:
    blocked = True
    print('Firewall correctly blocks unacknowledged ArmoRM PPO:', error)
assert blocked, 'FIREWALL BROKEN: ArmoRM PPO was not blocked without acknowledgement.'

# --- firewall: must WARN LOUDLY with the flag ---
firewall_ack = rs_ppo.check_reward_firewall('helpfulness', CONFIG['ARMORM_MODEL'],
                                            circular_armorm_acknowledged=True)
assert firewall_ack['circularity_acknowledged'] is True
assert 'RQ2 (proxy validity)' in firewall_ack['retired_research_questions']
print('Acknowledged circularity warning:', firewall_ack['warning'])


def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True, default=str) + '\n',
                    encoding='utf-8')
    assert path.exists(), f'Missing JSON output: {path}'


def write_numpy(path, values):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    np.save(path, np.asarray(values))
    assert path.exists(), f'Missing NumPy output: {path}'


PREREGISTRATION_PATH = OUTPUT_DIR / 'preregistration.json'
pre_registration = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'experiment': 'RS-faithful PPO against ArmoRM, deliberately circular upper-bound test',
    'seed': CONFIG['SEED'],
    'circularity': {
        'acknowledged': True,
        'retired_research_questions': ['RQ2 (proxy validity)'],
        'rationale': ('PPO reward model and evaluation model are both ArmoRM. This retires '
                      'proxy-validity claims but defines an UPPER-BOUND test: short-horizon PPO '
                      'against r_i makes Assumption 1 (delta ~ eta*grad r) approximately true by '
                      'construction, so this is the Best-Case regime for f(p,R). If f(p,R) does '
                      'not beat lambda=p here, it beats it nowhere.'),
    },
    'primary': {
        'metric': ('Delta U_p = mean over the 80 held-out prompts of '
                   'p^T ( r(theta(lambda*_M1+), prompt) - r(theta(p), prompt) ) -- the PAIRED '
                   'per-prompt difference on the native ArmoRM reward scale'),
        'success': 'mean Delta U_p > 0 with bootstrap 95% CI (2000 resamples) excluding 0',
        'bootstrap_n': CONFIG['BOOTSTRAP_N'],
        'bootstrap_seed': CONFIG['BOOTSTRAP_SEED'],
        'binding_run': True,
        'real_merges_required': True,
        'why_not_rank_normalised': (
            'The metric must be PAIRED and per-prompt to be bootstrappable over the 80 prompts. '
            'Rank-normalised U_p is a SET-LEVEL quantity (ranks taken over the search set B) and '
            'cannot be resampled prompt-wise. It is reported as a secondary level metric, '
            'together with a paired rank transform (ranks pooled over both models), which IS '
            'bootstrappable. Fixed BEFORE the binding run -- changing it afterwards = p-hacking.'),
    },
    'secondary_metrics': {
        'delta_U_p_rank': 'paired rank-normalised Delta U_p (ranks pooled over both models)',
        'delta_m_percent_gain': 'Delta m%(lambda*) - Delta m%(p); specialists = STL reference',
        'sign_test': 'one-sided sign test over preferences (v5 reference: 11/12, p=0.0032)',
        'portfolio_certificates': 'C1++/M1++/P2++/P3++ returning p = floor certificate',
    },
    'secondary_non_circular': {
        'R_minus_nonzero': 'yes/no',
        'floor_collapse': 'from LP: max_{1^T v = 0} min_i (Rv)_i <= tol',
        'wall_A_R2': 'per axis, linear vertex fit against true vertex rewards',
        'LMC': 'smooth yes/no along theta_SFT <-> theta_i',
    },
    'horizon_guard': {
        'requirement': 'reward curve must NOT plateau',
        'rationale': ('The upper-bound reading rests on delta ~ eta*grad r, i.e. a SHORT horizon. '
                      'If PPO converges, Assumption 1 breaks again and the Best-Case claim '
                      'loses its basis.'),
        'window': CONFIG['PLATEAU_WINDOW'],
        'eps': CONFIG['PLATEAU_SLOPE_EPS'],
    },
    'negative_control': 'quality-heavy preferences MUST run (Wall A predicts failure there)',
    'interpretation_rule': {
        'quality_success': 'Wall A was not a hard cap; strong positive result',
        'quality_failure': ('upper-bound negative result: endpoint-linear rules provably cannot '
                            'trace a nonlinear landscape, even in the Best-Case regime'),
    },
    'valid_claims': ['upper bound', 'R-minus', 'wall A (R2)', 'LMC'],
    'invalid_claims': ['proxy validity', 'generalization beyond this reward model'],
    'config': CONFIG,
}
write_json(PREREGISTRATION_PATH, pre_registration)
print('Wrote:', PREREGISTRATION_PATH)


## Phase 0 - VRAM, ArmoRM-Head-Verifikation, Batching-Beweis, Head-Sanity

Vier Pruefungen, jede eine direkte Lehre aus dem QRM-Debakel:

1. **VRAM** - ArmoRM 8B (4-bit) + TinyLlama-Policy + Value-Head + Referenz muessen gleichzeitig passen. Unter ~35 GB: harter Abbruch.
2. **F2 - Head-Index per Name.** `config.id2label` wird gelesen und jeder der fuenf Heads **namentlich** aufgeloest. Kein hartkodierter Index. Genau diese Zeile haette `verbosity == 0` bei QRM in Sekunde eins gefangen.
3. **F3 - Batching-Beweis.** `batched == single` wird bewiesen, die Padding-Seite empirisch aufgeloest. Schlaegt beides fehl -> Fallback auf `batch_size=1`.
4. **F5 - Sanity auf der richtigen Verteilung.** Die Heads werden auf **theta_SFT-Generierungen** geprueft (16-32 Tokens, identische Gen-Config wie PPO), *nicht* auf HelpSteer2-Referenzantworten. Ein Head kann auf Menschentext schoen streuen und auf kurzen TinyLlama-Rollouts kollabieren.

Faellt eine Assertion: **nicht trainieren.**


In [ ]:
HEAD_SANITY_PATH = OUTPUT_DIR / 'head_sanity.json'
SFT_MERGED = RS_RUNS_DIR / 'theta_sft' / 'merged'

if CONFIG['RUN_HEAD_SANITY']:
    from datasets import load_dataset
    from scipy.stats import pearsonr

    # ---- 1) VRAM -----------------------------------------------------------
    assert torch.cuda.is_available(), 'ArmoRM PPO path needs CUDA.'
    total_gb = torch.cuda.mem_get_info()[1] / 1024**3
    print(f'GPU: {torch.cuda.get_device_name(0)}  total={total_gb:.1f} GB')
    assert total_gb >= CONFIG['MIN_GPU_MEMORY_GB'], (
        f'A100-40GB-class runtime expected; got {total_gb:.1f} GB. '
        f'ArmoRM(4bit) + policy + ref + value head will not fit.')

    # ---- 2) F2: load scorer; index resolved BY NAME, asserted --------------
    scorer = rs_ppo.ArmoRMHeadScorer(axis='helpfulness', model_id=CONFIG['ARMORM_MODEL'])
    hs_indices = scorer.helpsteer_indices()
    resolved = {a: int(i) for a, i in zip(CONFIG['ATTRIBUTES'], hs_indices)}
    print('Resolved ArmoRM head indices (by name, not assumed):', resolved)
    assert len(set(hs_indices)) == 5, 'duplicate head indices resolved'

    # ---- 3) F3: prove batched == single ------------------------------------
    ds = load_dataset('nvidia/HelpSteer2', split=CONFIG['HEAD_SANITY_SPLIT'])
    rng = np.random.default_rng(CONFIG['SEED'])
    idx = rng.choice(len(ds), size=CONFIG['HEAD_SANITY_SAMPLES'], replace=False)
    rows = [ds[int(i)] for i in idx]
    prompts = [str(r['prompt']) for r in rows]
    labels = np.asarray([[float(r[a]) for a in CONFIG['ATTRIBUTES']] for r in rows],
                        dtype=np.float64)

    probe_pairs = [(p, 'This is a short probe response used to validate the scorer.')
                   for p in prompts[:8]]
    batching_report = scorer.validate_batching(probe_pairs)
    print('Batching report:', json.dumps(batching_report, indent=2))

    # ---- 4) F5: score the distribution PPO will actually produce -----------
    if CONFIG['HEAD_SANITY_ON_GENERATIONS']:
        assert SFT_MERGED.exists(), (
            'theta_SFT missing. Head sanity must run on POLICY generations, not on '
            'HelpSteer2 reference responses. Run the SFT phase first, or set '
            "CONFIG['HEAD_SANITY_ON_GENERATIONS']=False and accept that the sanity "
            'check then measures the WRONG distribution.')
        responses = rs_ppo.generate_responses(str(SFT_MERGED), prompts, seed=CONFIG['SEED'])
        distribution = 'theta_SFT generations (16-32 tokens, PPO gen-config)'
    else:
        responses = [str(r['response']) for r in rows]
        distribution = 'HelpSteer2 reference responses (WRONG distribution - diagnostic only)'
    print('Sanity distribution:', distribution)

    scores = scorer.score_all_heads(prompts, responses)
    assert scores.shape == (CONFIG['HEAD_SANITY_SAMPLES'], 5)
    assert np.all(np.isfinite(scores)), 'non-finite ArmoRM scores'

    # ---- degeneracy assertions (the QRM lesson) ----------------------------
    stds = scores.std(axis=0)
    uniques = [int(len(np.unique(scores[:, i]))) for i in range(5)]
    failures = []
    for i, axis in enumerate(CONFIG['ATTRIBUTES']):
        if stds[i] <= CONFIG['HEAD_STD_MIN']:
            failures.append(f'{axis}: constant head (std={stds[i]:.3e})')
        if uniques[i] <= CONFIG['HEAD_UNIQUE_MIN']:
            failures.append(f'{axis}: only {uniques[i]} unique values')

    # ---- axis-discriminativeness (diag of ArmoRM x HelpSteer2 labels) ------
    cross = np.full((5, 5), np.nan)
    for i in range(5):
        for j in range(5):
            if scores[:, i].std() > 0 and labels[:, j].std() > 0:
                cross[i, j] = float(pearsonr(scores[:, i], labels[:, j]).statistic)
    diag = np.diag(cross)
    row_argmax_on_diag = [bool(np.nanargmax(cross[i]) == i) for i in range(5)]
    n_discriminative = int(sum(row_argmax_on_diag))
    print(f'Axis-discriminative heads (row argmax on own axis): {n_discriminative}/5')

    passed = len(failures) == 0
    head_sanity = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'n': int(scores.shape[0]),
        'distribution': distribution,
        'on_generations': bool(CONFIG['HEAD_SANITY_ON_GENERATIONS']),
        'attributes': CONFIG['ATTRIBUTES'],
        'resolved_head_indices': resolved,
        'batching_report': batching_report,
        'std': dict(zip(CONFIG['ATTRIBUTES'], stds.tolist())),
        'unique_counts': dict(zip(CONFIG['ATTRIBUTES'], uniques)),
        'armorm_vs_helpsteer2_pearson': cross.tolist(),
        'diagonal': dict(zip(CONFIG['ATTRIBUTES'], diag.tolist())),
        'n_axis_discriminative': n_discriminative,
        'failures': failures,
        'passed': passed,
        'circularity_acknowledged': True,
    }
    write_json(HEAD_SANITY_PATH, head_sanity)
    write_numpy(OUTPUT_DIR / 'head_sanity_scores.npy', scores)
    write_numpy(OUTPUT_DIR / 'head_sanity_labels.npy', labels)
    display(pd.DataFrame(cross, index=CONFIG['ATTRIBUTES'], columns=CONFIG['ATTRIBUTES']))

    del scorer
    torch.cuda.empty_cache()

    assert passed, 'HEAD SANITY FAILED -> DO NOT TRAIN:\n  ' + '\n  '.join(failures)
    print('Head sanity PASSED. Safe to train.')
    if n_discriminative < 5:
        print(f'NOTE: only {n_discriminative}/5 heads are axis-discriminative. '
              f'Not a blocker, but report this honestly (v5 found 2/5 self-maximizing).')
else:
    write_json(HEAD_SANITY_PATH, {'passed': False, 'skipped': True,
                                  'reason': 'RUN_HEAD_SANITY is False'})
    print('Head sanity skipped.')


## Phase 1 und 2 - SFT-Basis und die fuenf PPO-Laeufe

**F1:** Beide Phasen laufen **in-process** ueber `rs_ppo.run_sft()` / `rs_ppo.run_ppo(...)`. Damit greifen `out_dir` (-> `OUTPUT_DIR/rs_runs`), `batch_size`, `total_ppo_steps`, `n_prompts` und 4-bit wirklich.

**Equal-N:** identischer `prompt_seed=137`, `train_seed=911`, identische Schrittzahl fuer **alle** Achsen. Kein per-Achsen-Tuning (Betreuer: "Training ist Nebensache").

**F6:** nach jedem Lauf wird `reward_plateaued` geprueft. Plateau = auskonvergiert = Assumption 1 gebrochen = Upper-Bound-Argument weg.


In [ ]:
PLATEAU_PATH = OUTPUT_DIR / 'plateau_report.json'

if CONFIG['RUN_SFT']:
    print('=== Phase 1: SFT -> theta_SFT (shared init for all PPO runs) ===')
    sft_path = rs_ppo.run_sft()
    print('theta_SFT:', sft_path)
else:
    print('RUN_SFT is False; SFT phase not started.')
    if SFT_MERGED.exists():
        print('  (existing theta_SFT found at', SFT_MERGED, ')')

if CONFIG['RUN_PPO']:
    head_sanity = json.loads(HEAD_SANITY_PATH.read_text(encoding='utf-8'))
    assert head_sanity.get('passed') is True, 'Head sanity did not pass; DO NOT TRAIN.'
    assert SFT_MERGED.exists(), 'theta_SFT missing; run the SFT phase first.'

    plateau_report = {}
    for axis in CONFIG['PPO_AXES']:
        print(f'\n=== Phase 2: PPO run for axis {axis!r} (CIRCULAR: reward = ArmoRM) ===')
        result = rs_ppo.run_ppo(
            axis,
            reward_model_id=CONFIG['ARMORM_MODEL'],
            circular_armorm_acknowledged=CONFIG['CIRCULAR_ARMORM_ACKNOWLEDGED'],
        )
        plateau_report[axis] = result['plateau']
        adapter_dir = RS_RUNS_DIR / f'ppo_{axis}' / 'adapter'
        assert adapter_dir.is_dir(), f'adapter not written to expected path: {adapter_dir}'
        print(f'  -> adapter at {adapter_dir}')

    plateaued = [a for a, p in plateau_report.items() if p.get('reward_plateaued') is True]
    plateau_report['_summary'] = {
        'axes_plateaued': plateaued,
        'upper_bound_reading_valid': len(plateaued) == 0,
        'note': ('The upper-bound claim requires a SHORT horizon on every axis. Any axis that '
                 'plateaued has converged -> delta != eta*grad r -> Assumption 1 broken there.'),
    }
    write_json(PLATEAU_PATH, plateau_report)
    if plateaued:
        print(f'\nWARNING: axes plateaued: {plateaued}. The Best-Case/upper-bound reading '
              f'is NOT valid for these axes. Report this, do not hide it.')
    else:
        print('\nNo axis plateaued -> short-horizon regime intact; upper-bound reading holds.')
else:
    write_json(PLATEAU_PATH, {'pending': True, 'reason': 'RUN_PPO is False'})
    print('RUN_PPO is False. Planned in-process calls:')
    for axis in CONFIG['PPO_AXES']:
        print(f"  rs_ppo.run_ppo({axis!r}, reward_model_id={CONFIG['ARMORM_MODEL']!r}, "
              f"circular_armorm_acknowledged=True)")


## Phase 3 - Geometrie: delta-Normen, R, R-minus, Floor-LP

**Nicht zirkulaer** (0 Reward-Queries). Die zentrale Frage: **bricht PPO die konfliktfreie Geometrie?** Auf HelpSteer2/SFT war `R_cos` off-diag komplett positiv (0.299-0.397) -> `R- = 0` -> Floor-Kollaps `F_p = {p}` -> C1++/M1++/P2++/P3++ geben alle `p` zurueck.

**F7:** Floor-Kollaps wird jetzt **aus dem LP** entschieden (`max_{1^T v = 0, ||v||_1 <= 1} min_i (Rv)_i <= tol`), nicht aus dem Vorzeichen der Off-Diagonalen. Das LP *ist* das Kriterium; `R- != 0` war nur eine Heuristik dafuer.


In [ ]:
DELTA_NORMS_PATH = OUTPUT_DIR / 'delta_norms.json'
GEOMETRY_PRECHECK_PATH = OUTPUT_DIR / 'geometry_precheck.json'
R_GRAM_PATH = OUTPUT_DIR / 'R_gram.npy'
R_COS_PATH = OUTPUT_DIR / 'R_cos.npy'


def floor_lp(R, tol):
    """max_{1^T v = 0, ||v||_1 <= 1} min_i (Rv)_i.  Floor collapses iff value <= tol."""
    from scipy.optimize import linprog
    n = R.shape[0]
    # variables: [x+ (n), x- (n), t] with v = x+ - x-
    c = np.zeros(2 * n + 1); c[-1] = -1.0                      # maximise t
    A_ub, b_ub = [], []
    for i in range(n):                                          # t <= (R v)_i
        row = np.zeros(2 * n + 1)
        row[:n] = -R[i]; row[n:2 * n] = R[i]; row[-1] = 1.0
        A_ub.append(row); b_ub.append(0.0)
    row = np.zeros(2 * n + 1); row[:2 * n] = 1.0                # ||v||_1 <= 1
    A_ub.append(row); b_ub.append(1.0)
    A_eq = np.asarray([np.r_[np.ones(n), -np.ones(n), 0.0]])    # 1^T v = 0
    lp = linprog(c, A_ub=np.asarray(A_ub), b_ub=np.asarray(b_ub),
                 A_eq=A_eq, b_eq=np.asarray([0.0]),
                 bounds=[(0.0, None)] * (2 * n) + [(None, None)], method='highs')
    assert lp.success, lp.message
    value = float(-lp.fun)
    return value, bool(value <= tol)


if CONFIG['RUN_GEOMETRY']:
    from src.effective_lora_geometry import (effective_lora_inner_product,
                                             load_effective_lora_geometry,
                                             validate_compatible_geometries)
    adapter_paths = {a: RS_RUNS_DIR / f'ppo_{a}' / 'adapter' for a in CONFIG['ATTRIBUTES']}
    missing = [str(p) for p in adapter_paths.values() if not p.is_dir()]
    assert not missing, 'Missing PPO adapters: ' + ', '.join(missing)

    geometries = {a: load_effective_lora_geometry(p) for a, p in adapter_paths.items()}
    validate_compatible_geometries([geometries[a] for a in CONFIG['ATTRIBUTES']],
                                   CONFIG['ATTRIBUTES'])

    n = len(CONFIG['ATTRIBUTES'])
    gram = np.zeros((n, n))
    for i, left in enumerate(CONFIG['ATTRIBUTES']):
        for j, right in enumerate(CONFIG['ATTRIBUTES']):
            gram[i, j] = effective_lora_inner_product(geometries[left], geometries[right])
    gram = 0.5 * (gram + gram.T)
    norms = np.sqrt(np.maximum(np.diag(gram), 0.0))
    assert np.all(norms > 0), 'a PPO adapter has a zero task vector (no learning happened)'
    cos = gram / np.outer(norms, norms)
    np.fill_diagonal(cos, 1.0)

    offdiag = cos[~np.eye(n, dtype=bool)]
    neg = offdiag[offdiag < 0]

    # F7: the LP IS the floor criterion
    lp_value, floor_collapsed = floor_lp(cos, CONFIG['FLOOR_TOL'])

    write_numpy(R_GRAM_PATH, gram)
    write_numpy(R_COS_PATH, cos)
    write_json(DELTA_NORMS_PATH, {
        'attributes': CONFIG['ATTRIBUTES'],
        'delta_norm': dict(zip(CONFIG['ATTRIBUTES'], norms.tolist())),
        'mean_norm': float(norms.mean()),
        'max_percent_deviation_from_mean': float(np.max(np.abs(norms / norms.mean() - 1.0)) * 100),
        'note': 'Equal-N should keep these uniform (SFT reference: 5.33-5.46).',
    })
    write_json(GEOMETRY_PRECHECK_PATH, {
        'attributes': CONFIG['ATTRIBUTES'],
        'R_minus_nonzero': bool(len(neg) > 0),
        'negative_offdiag_count': int(len(neg)),
        'negative_offdiag_min': float(neg.min()) if len(neg) else None,
        'cosine_offdiag_min': float(offdiag.min()),
        'cosine_offdiag_max': float(offdiag.max()),
        'floor_lp_max_min_Rv': lp_value,
        'floor_lp_l1_bound': 1.0,
        'floor_collapsed': floor_collapsed,       # F7: THE criterion
        'floor_tol': CONFIG['FLOOR_TOL'],
        'interpretation': ('floor_collapsed=True  -> F_p = {p}: C1++/M1++/P2++/P3++ all return p '
                           '(report as certificates, not failures); only M1+ can move. '
                           'floor_collapsed=False -> PPO broke the conflict-free geometry; the '
                           'vector-safe portfolio becomes non-trivial for the first time.'),
        'circular_do_not_report_as_proxy_validation': True,
    })
    display(pd.DataFrame(cos, index=CONFIG['ATTRIBUTES'], columns=CONFIG['ATTRIBUTES']))
    print(f'\nR-minus nonzero: {len(neg) > 0}  (negative off-diagonals: {len(neg)})')
    print(f'Floor LP value  : {lp_value:.6e}')
    print(f'FLOOR COLLAPSED : {floor_collapsed}  -> ' +
          ('F_p = {p}; vector-safe methods return p.' if floor_collapsed
           else 'floor is OPEN; vector-safe methods can move.'))
else:
    write_json(DELTA_NORMS_PATH, {'pending': True, 'reason': 'RUN_GEOMETRY is False'})
    write_json(GEOMETRY_PRECHECK_PATH, {'pending': True, 'reason': 'RUN_GEOMETRY is False'})
    print('RUN_GEOMETRY is False; geometry skipped.')


## Phase 4a - Reward-Collection ueber die Suchmenge B (F8)

Der teuerste Schritt, und die Voraussetzung fuer **alles** danach: fuer jedes `lambda` in B wird `theta(lambda)` **echt gemergt**, ueber 80 held-out Prompts generiert und mit allen fuenf ArmoRM-Heads gescort. Ergebnis: `reward_matrix.npy` mit Shape `(|B|, 5)`.

Prompts: `validation`-Slice ab Offset **160** - disjunkt zum v5-Konfirmations-Slice `[80:160]`.

Zeilen 0-4 von B sind die Vertices (= die fuenf PPO-Spezialisten) und liefern die STL-Referenz fuer `Delta m%`.


In [ ]:
from src.proxy_validation import build_search_set

SEARCH_SET_PATH = OUTPUT_DIR / 'search_set_B.npy'
REWARD_MATRIX_PATH = OUTPUT_DIR / 'reward_matrix.npy'
REWARD_PROMPTS_PATH = OUTPUT_DIR / 'reward_prompts.json'

PREFERENCES = {
    # quality-heavy = PRE-REGISTERED NEGATIVE CONTROL (Wall A predicts failure here)
    'dominant_helpfulness': [0.5, 0.125, 0.125, 0.125, 0.125],
    'dominant_correctness': [0.125, 0.5, 0.125, 0.125, 0.125],
    'dominant_coherence':   [0.125, 0.125, 0.5, 0.125, 0.125],
    'only_helpfulness':     [1.0, 0.0, 0.0, 0.0, 0.0],
    'only_correctness':     [0.0, 1.0, 0.0, 0.0, 0.0],
    'only_coherence':       [0.0, 0.0, 1.0, 0.0, 0.0],
    # compl/verb = the regime where geometry worked in v5
    'dominant_complexity':  [0.125, 0.125, 0.125, 0.5, 0.125],
    'dominant_verbosity':   [0.125, 0.125, 0.125, 0.125, 0.5],
    'only_complexity':      [0.0, 0.0, 0.0, 1.0, 0.0],
    'only_verbosity':       [0.0, 0.0, 0.0, 0.0, 1.0],
    'uniform':              [0.2, 0.2, 0.2, 0.2, 0.2],
}
QUALITY_PREFS = ['dominant_helpfulness', 'dominant_correctness', 'dominant_coherence',
                 'only_helpfulness', 'only_correctness', 'only_coherence']
CV_PREFS = ['dominant_complexity', 'dominant_verbosity', 'only_complexity', 'only_verbosity']

B = build_search_set(5, n_dirichlet=CONFIG['SEARCH_SET_DIRICHLET'],
                     preferences=list(PREFERENCES.values()),
                     seed=CONFIG['SEARCH_SET_SEED'])
write_numpy(SEARCH_SET_PATH, B)
print('Search set B:', B.shape, '(rows 0-4 = vertices = the five PPO specialists)')
assert np.allclose(B[:5], np.eye(5)), 'rows 0-4 of B must be the vertices'


def merge_theta(lmbda, adapter_paths, base_path):
    """theta(lambda) = theta_SFT + sum_i lambda_i * delta_i, realised on LoRA weights."""
    from peft import PeftModel
    from transformers import AutoModelForCausalLM
    model = AutoModelForCausalLM.from_pretrained(str(base_path), torch_dtype=torch.bfloat16,
                                                 device_map='auto')
    names = list(adapter_paths.keys())
    peft_model = PeftModel.from_pretrained(model, str(adapter_paths[names[0]]),
                                           adapter_name=names[0])
    for name in names[1:]:
        peft_model.load_adapter(str(adapter_paths[name]), adapter_name=name)
    weights = [float(w) for w in lmbda]
    peft_model.add_weighted_adapter(adapters=names, weights=weights,
                                    adapter_name='merged', combination_type='linear')
    peft_model.set_adapter('merged')
    merged = peft_model.merge_and_unload()
    merged.eval()
    return merged


if CONFIG['RUN_REWARD_COLLECTION']:
    from datasets import load_dataset
    from transformers import AutoTokenizer

    adapter_paths = {a: RS_RUNS_DIR / f'ppo_{a}' / 'adapter' for a in CONFIG['ATTRIBUTES']}
    for p in adapter_paths.values():
        assert p.is_dir(), f'missing adapter: {p}'
    assert SFT_MERGED.exists(), 'theta_SFT missing'

    ds = load_dataset('nvidia/HelpSteer2', split=CONFIG['REWARD_PROMPT_SPLIT'])
    off = CONFIG['REWARD_PROMPT_OFFSET']
    eval_prompts = [str(ds[i]['prompt'])
                    for i in range(off, off + CONFIG['REWARD_NUM_PROMPTS'])]
    write_json(REWARD_PROMPTS_PATH, {'split': CONFIG['REWARD_PROMPT_SPLIT'],
                                     'offset': off, 'n': len(eval_prompts),
                                     'disjoint_from_v5_confirmatory_slice': '[80:160]',
                                     'prompts': eval_prompts})

    scorer = rs_ppo.ArmoRMHeadScorer(axis='helpfulness', model_id=CONFIG['ARMORM_MODEL'])
    scorer.validate_batching([(p, 'probe response') for p in eval_prompts[:8]])

    tok = AutoTokenizer.from_pretrained(str(SFT_MERGED))
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = 'left'

    reward_rows = []
    t0 = time.time()
    for k, lmbda in enumerate(B):
        rs_ppo.set_all_seeds(CONFIG['SEED'])          # same rollouts for every lambda
        model = merge_theta(lmbda, adapter_paths, SFT_MERGED)
        responses = []
        for start in range(0, len(eval_prompts), 8):
            chunk = eval_prompts[start:start + 8]
            texts = [tok.apply_chat_template([{'role': 'user', 'content': p}],
                                             tokenize=False, add_generation_prompt=True)
                     for p in chunk]
            enc = tok(texts, return_tensors='pt', padding=True).to(model.device)
            n_new = random.randint(rs_ppo.CFG['output_min_len'], rs_ppo.CFG['output_max_len'])
            with torch.inference_mode():
                out = model.generate(**enc, max_new_tokens=n_new, do_sample=True,
                                     top_k=0, top_p=1.0, pad_token_id=tok.eos_token_id)
            responses.extend(tok.batch_decode(out[:, enc['input_ids'].shape[1]:],
                                              skip_special_tokens=True))
        heads = scorer.score_all_heads(eval_prompts, responses)   # [80, 5]
        reward_rows.append(heads.mean(axis=0))
        del model
        torch.cuda.empty_cache()
        if (k + 1) % 5 == 0 or k == len(B) - 1:
            el = time.time() - t0
            print(f'  lambda {k+1}/{len(B)}  elapsed={el/60:.1f} min  '
                  f'eta={(el/(k+1)*(len(B)-k-1))/60:.1f} min')

    Reward = np.asarray(reward_rows, dtype=np.float64)
    assert Reward.shape == (B.shape[0], 5)
    assert np.all(np.isfinite(Reward))
    # cache-contamination guard (v4 lesson): vertex rows must differ from one another
    assert len(np.unique(np.round(Reward[:5], 8), axis=0)) == 5, \
        'vertex reward rows are identical -> cache contamination or merge is a no-op'
    write_numpy(REWARD_MATRIX_PATH, Reward)
    print('reward_matrix.npy written:', Reward.shape)
    display(pd.DataFrame(Reward[:5], index=CONFIG['ATTRIBUTES'], columns=CONFIG['ATTRIBUTES']))
    del scorer
    torch.cuda.empty_cache()
else:
    print('RUN_REWARD_COLLECTION is False; no reward matrix built (no fake numbers).')


## Phase 4b - Wall-A-Test (R2) und LMC

**Wall A** ist der eigentliche Go/No-Go. Linearer Vertex-Fit pro Achse gegen die *echten* Vertex-Rewards, `R2` auf den Nicht-Vertex-Punkten von B. Bleibt `quality R2 ~ 0` oder negativ, kann **keine** endpoint-lineare Regel dort etwas ausrichten - weder `R` noch ein gemessenes `H`. Das gilt unabhaengig davon, wie gut das Training lief, und ist **nicht zirkulaer**: Nichtlinearitaet ist eine Eigenschaft von ArmoRM x Interpolation.

**LMC** (RS Working Hyp. 1): Reward entlang `theta_SFT <-> theta_i`. RS warnt selbst, dass bei antagonistischen Rewards die lineare Mode-Connectivity brechen kann - die Grundvoraussetzung der ganzen Interpolationsfamilie.


In [ ]:
LINEARITY_R2_PATH = OUTPUT_DIR / 'linearity_r2.json'
LMC_CHECK_PATH = OUTPUT_DIR / 'lmc_check.json'

# ---- Wall A -----------------------------------------------------------------
if REWARD_MATRIX_PATH.exists():
    Reward = np.load(REWARD_MATRIX_PATH)
    assert Reward.shape == (B.shape[0], 5)
    vertex_rewards = Reward[:5]                      # [vertex k, axis j]
    mask = np.arange(len(B)) >= 5                    # evaluate off the vertices

    rows = []
    for j, axis in enumerate(CONFIG['ATTRIBUTES']):
        y = Reward[:, j]
        y_hat = B @ vertex_rewards[:, j]             # endpoint-linear prediction
        ss_res = float(np.sum((y[mask] - y_hat[mask]) ** 2))
        ss_tot = float(np.sum((y[mask] - y[mask].mean()) ** 2))
        r2 = float(1.0 - ss_res / ss_tot) if ss_tot > 0 else None
        rows.append({'axis': axis, 'r2_vertex_linear_fit': r2,
                     'regime': 'quality' if axis in ('helpfulness', 'correctness', 'coherence')
                               else 'complexity/verbosity'})
    quality_r2 = [r['r2_vertex_linear_fit'] for r in rows if r['regime'] == 'quality'
                  and r['r2_vertex_linear_fit'] is not None]
    wall_a_stands = bool(quality_r2 and max(quality_r2) < 0.3)

    write_json(LINEARITY_R2_PATH, {
        'circular_do_not_report_as_proxy_validation': True,
        'rows': rows,
        'quality_r2_max': float(max(quality_r2)) if quality_r2 else None,
        'wall_a_stands': wall_a_stands,
        'interpretation': ('wall_a_stands=True: U_p is nonlinear in lambda on the quality axes. '
                           'No endpoint-linear rule (R-based or H-based) can trace it -- even in '
                           'this Best-Case circular regime. This is the upper-bound negative '
                           'result and it is thesis-bearing. '
                           'wall_a_stands=False: Wall A was not a hard cap; strong positive.'),
        'v5_reference': 'SFT/HelpSteer2: quality R2 ~ 0 or negative; compl/verb up to 0.77',
    })
    display(pd.DataFrame(rows))
    print(f'\nWALL A STANDS: {wall_a_stands}')
else:
    write_json(LINEARITY_R2_PATH, {'pending': True,
                                   'reason': 'reward_matrix.npy missing; run RUN_REWARD_COLLECTION'})
    print('Wall A pending: no reward_matrix.npy.')

# ---- LMC --------------------------------------------------------------------
if CONFIG['RUN_LMC']:
    from datasets import load_dataset
    from transformers import AutoTokenizer

    adapter_paths = {a: RS_RUNS_DIR / f'ppo_{a}' / 'adapter' for a in CONFIG['ATTRIBUTES']}
    ds = load_dataset('nvidia/HelpSteer2', split=CONFIG['REWARD_PROMPT_SPLIT'])
    off = CONFIG['REWARD_PROMPT_OFFSET']
    eval_prompts = [str(ds[i]['prompt']) for i in range(off, off + 32)]   # LMC: 32 prompts suffice

    scorer = rs_ppo.ArmoRMHeadScorer(axis='helpfulness', model_id=CONFIG['ARMORM_MODEL'])
    scorer.validate_batching([(p, 'probe response') for p in eval_prompts[:8]])
    tok = AutoTokenizer.from_pretrained(str(SFT_MERGED))
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = 'left'

    curves = {}
    for i, axis in enumerate(CONFIG['ATTRIBUTES']):
        curve = []
        for t in CONFIG['LMC_GRID']:
            lmbda = np.zeros(5); lmbda[i] = t       # theta_SFT (t=0) -> theta_i (t=1)
            rs_ppo.set_all_seeds(CONFIG['SEED'])
            model = merge_theta(lmbda, adapter_paths, SFT_MERGED)
            responses = []
            for start in range(0, len(eval_prompts), 8):
                chunk = eval_prompts[start:start + 8]
                texts = [tok.apply_chat_template([{'role': 'user', 'content': p}],
                                                 tokenize=False, add_generation_prompt=True)
                         for p in chunk]
                enc = tok(texts, return_tensors='pt', padding=True).to(model.device)
                with torch.inference_mode():
                    out = model.generate(**enc, max_new_tokens=24, do_sample=True,
                                         top_k=0, top_p=1.0, pad_token_id=tok.eos_token_id)
                responses.extend(tok.batch_decode(out[:, enc['input_ids'].shape[1]:],
                                                  skip_special_tokens=True))
            heads = scorer.score_all_heads(eval_prompts, responses)
            curve.append(float(heads.mean(axis=0)[i]))   # own-axis reward
            del model; torch.cuda.empty_cache()
        curves[axis] = curve
        print(f'[lmc] {axis}: {[round(c, 4) for c in curve]}')

    # RS Working Hyp. 1: reward along the path >= linear interpolation of endpoints
    lmc = {}
    for axis, curve in curves.items():
        lo, hi = curve[0], curve[-1]
        lin = [lo + t * (hi - lo) for t in CONFIG['LMC_GRID']]
        deficits = [c - l for c, l in zip(curve, lin)]
        lmc[axis] = {'grid': CONFIG['LMC_GRID'], 'reward': curve,
                     'linear_interp': lin, 'deficit': deficits,
                     'min_deficit': float(min(deficits)),
                     'lmc_holds': bool(min(deficits) >= -0.01)}
    lmc['_summary'] = {
        'lmc_holds_all_axes': all(v['lmc_holds'] for k, v in lmc.items() if not k.startswith('_')),
        'note': ('RS Appendix B.3.2 warns that fully antagonistic rewards can break linear mode '
                 'connectivity -- and LMC is the precondition of the whole interpolation family.'),
    }
    write_json(LMC_CHECK_PATH, lmc)
    print('\nLMC holds on all axes:', lmc['_summary']['lmc_holds_all_axes'])
    del scorer; torch.cuda.empty_cache()
else:
    write_json(LMC_CHECK_PATH, {'pending': True, 'reason': 'RUN_LMC is False'})
    print('RUN_LMC is False; LMC skipped.')


## Phase 5 - Der bindende Test: M1+ vs. lambda=p mit echten Merges (F8)

Der **primaere vorregistrierte Endpunkt**. Fuer jede Praeferenz `p`:

1. `lambda* = M1+(p, R_cos)` via SLSQP (`rho=0.5`).
2. **Echt mergen** - `theta(lambda*)` und `theta(p)`. Keine Nearest-Point-in-B-Approximation.
3. `Delta U_p = U_p(lambda*) - U_p(p)` mit **rang-normalisiertem** `U_p` (min-max waere zirkulaer, s. Metrik-Audit).
4. **Bootstrap-95%-CI** ueber die 80 Prompts (2000 Resamples). Ein Effekt zaehlt nur, wenn das CI die 0 ausschliesst.
5. `Delta m%`-Gewinn (`lambda*` minus `p`) gegen die Spezialisten-Referenz.

**Alle fuenf Methoden laufen** - M1+, C1++ (mit Trust-Region, wie in der Portfolio-Definition), M1++, P2++, P3++. Geben die vier vektorsicheren `p` zurueck, ist das das **Floor-Zertifikat** ("keine Richtung verbessert alle Objectives gleichzeitig") und wird als Theorem berichtet, nicht als Versagen.

**Metrik-Korrektur:** Der primaere Endpunkt ist der **gepaarte Per-Prompt-Delta** auf der ArmoRM-Rohskala - nur der laesst sich ueber die 80 Prompts bootstrappen. Rang-normalisiertes `U_p` ist eine **Set-Level**-Groesse (Raenge ueber B) und prompt-weise nicht resamplebar; es laeuft als **sekundaere** Metrik mit, zusammen mit einem gepaarten Rang-Transform (Raenge ueber beide Modelle gepoolt), der bootstrappbar ist. Beide **vor** dem bindenden Lauf festgelegt.

**Negativkontrolle:** die quality-lastigen Praeferenzen laufen zwingend mit.


In [ ]:
MERGE_RESULTS_PATH = OUTPUT_DIR / 'merge_results.json'

# ============================================================================
# FULL PORTFOLIO -- C1++, M1+, M1++, P2++, P3++
# g_i := R[:, i] is the reward gradient of objective i in coefficient space.
# Delta(lam) = R(lam - p);  floor F_p = {lam in simplex : Delta_i(lam) >= 0 for all i}
# ============================================================================
from scipy.optimize import minimize
from scipy.stats import rankdata, binomtest

PI0 = lambda m: np.eye(m) - np.ones((m, m)) / m       # simplex tangent projector


def improvements(p, R, lam):
    return R @ (np.asarray(lam, float) - np.asarray(p, float))


def line_search_into_floor(p, R, d, tol=1e-10):
    """Largest t >= 0 with p + t*d in F_p AND in the simplex. Requires 1^T d = 0.

    NOTE: the projection Pi0 does NOT preserve common ascent (that is MGDA's property
    of d_M itself, not of its projection). The line search is what actually guarantees
    floor membership -- with conservative fallback t=0 -> lambda = p.
    """
    p = np.asarray(p, float); d = np.asarray(d, float)
    assert abs(d.sum()) < 1e-8, 'direction must lie in the simplex tangent space'
    if np.any(R @ d < -tol) or np.linalg.norm(d) < tol:
        return 0.0
    ts = [np.inf] + [-p[i] / d[i] for i in range(len(p)) if d[i] < -tol]
    t = float(min(ts))
    return 0.0 if not np.isfinite(t) else max(0.0, t)


def m1_plus(p, R, rho):
    """argmax_{lam in simplex}  p'R lam - rho (lam-p)'R(lam-p)    [SCALAR-safe only]"""
    p = np.asarray(p, float); n = len(p)
    obj = lambda x: -(p @ R @ x - rho * (x - p) @ R @ (x - p))
    jac = lambda x: -(R @ p - 2 * rho * R @ (x - p))
    res = minimize(obj, p.copy(), jac=jac, method='SLSQP', bounds=[(0., 1.)] * n,
                   constraints=[{'type': 'eq', 'fun': lambda x: x.sum() - 1.,
                                 'jac': lambda x: np.ones(n)}],
                   options={'maxiter': 500, 'ftol': 1e-12})
    assert res.success, res.message
    x = np.clip(res.x, 0., None)
    return x / x.sum()


def c1_plus_plus(p, R, c, eps):
    """max_{lam in simplex} min_i Delta_i(lam)  s.t. (lam-p)'R(lam-p) <= c^2 max(p'Rp, eps)

    WITH the trust region -- that is the portfolio definition. Dropping it (LP-only)
    keeps floor membership but LOSES the faithfulness / bounded-excursion bound.
    Intrinsic guarantee: p is feasible with value 0  =>  t* >= 0  =>  lam in F_p.
    """
    p = np.asarray(p, float); n = len(p)
    radius2 = (c ** 2) * max(float(p @ R @ p), eps)
    obj = lambda z: -z[-1]
    jac = lambda z: np.r_[np.zeros(n), -1.0]
    cons = [
        {'type': 'eq', 'fun': lambda z: z[:n].sum() - 1.,
         'jac': lambda z: np.r_[np.ones(n), 0.]},
        {'type': 'ineq', 'fun': lambda z: R @ (z[:n] - p) - z[-1],
         'jac': lambda z: np.hstack([R, -np.ones((n, 1))])},
        {'type': 'ineq', 'fun': lambda z: radius2 - (z[:n] - p) @ R @ (z[:n] - p),
         'jac': lambda z: np.r_[-2 * R @ (z[:n] - p), 0.]},
    ]
    res = minimize(obj, np.r_[p, 0.0], jac=jac, method='SLSQP', constraints=cons,
                   bounds=[(0., 1.)] * n + [(None, None)],
                   options={'maxiter': 800, 'ftol': 1e-12})
    if not res.success:
        return p.copy(), 0.0
    lam = np.clip(res.x[:n], 0., None); lam = lam / lam.sum()
    t_star = float(min(improvements(p, R, lam)))
    if t_star < -1e-8:
        return p.copy(), 0.0
    return lam, max(t_star, 0.0)


def m1_plus_plus(p, R):
    """MGDA min-norm direction d_M = R alpha*, projected, then line-searched into F_p."""
    p = np.asarray(p, float); n = len(p)
    obj = lambda a: float(np.dot(R @ a, R @ a))
    jac = lambda a: 2 * R.T @ (R @ a)
    res = minimize(obj, np.ones(n) / n, jac=jac, method='SLSQP', bounds=[(0., 1.)] * n,
                   constraints=[{'type': 'eq', 'fun': lambda a: a.sum() - 1.,
                                 'jac': lambda a: np.ones(n)}],
                   options={'maxiter': 500, 'ftol': 1e-14})
    if not res.success:
        return p.copy(), 0.0
    d = PI0(n) @ (R @ res.x)
    t = line_search_into_floor(p, R, d)
    return (p + t * d if t > 0 else p.copy()), t


def p2_plus_plus(p, R):
    """PCGrad surgery on g_i = R[:, i]; deterministic order (strongest conflict first)."""
    p = np.asarray(p, float); n = len(p)
    G = [R[:, i].astype(float).copy() for i in range(n)]
    pairs = sorted([(i, j) for i in range(n) for j in range(n) if i != j],
                   key=lambda ij: float(R[:, ij[0]] @ R[:, ij[1]]))
    fired = 0
    for i, j in pairs:
        dot = float(G[i] @ G[j])
        if dot < 0:
            G[i] = G[i] - (dot / float(G[j] @ G[j])) * G[j]
            fired += 1
    d = PI0(n) @ sum(p[i] * G[i] for i in range(n))
    t = line_search_into_floor(p, R, d)
    return (p + t * d if t > 0 else p.copy()), t, fired


def p3_plus_plus(p, R, n_starts=8, seed=137):
    """min_{lam in F_p} lam' R^- lam (conflict energy). Non-convex -> multi-start SLSQP."""
    p = np.asarray(p, float); n = len(p)
    Rm = np.maximum(0.0, -R.copy()); np.fill_diagonal(Rm, 0.0)
    if not np.any(Rm > 0):
        return p.copy(), 0.0, True                # C == 0 everywhere -> indifferent
    cons = [{'type': 'eq', 'fun': lambda x: x.sum() - 1., 'jac': lambda x: np.ones(n)},
            {'type': 'ineq', 'fun': lambda x: R @ (x - p), 'jac': lambda x: R}]
    obj = lambda x: float(x @ Rm @ x)
    rng_p3 = np.random.default_rng(seed)
    best, best_val = p.copy(), obj(p)
    for x0 in [p.copy()] + [rng_p3.dirichlet(np.ones(n)) for _ in range(n_starts - 1)]:
        r = minimize(obj, x0, jac=lambda x: 2 * Rm @ x, method='SLSQP', constraints=cons,
                     bounds=[(0., 1.)] * n, options={'maxiter': 500, 'ftol': 1e-12})
        if r.success:
            x = np.clip(r.x, 0., None); x = x / x.sum()
            if np.all(improvements(p, R, x) >= -1e-7) and obj(x) < best_val - 1e-12:
                best, best_val = x, obj(x)
    return best, best_val, False


def run_portfolio(p, R, cfg):
    """All five mappings. The four vector-safe ones returning p IS the floor certificate."""
    p = np.asarray(p, float)
    lam_m1p = m1_plus(p, R, cfg['M1PLUS_RHO'])
    lam_c1, t_c1 = c1_plus_plus(p, R, cfg['C1PP_C'], cfg['C1PP_EPS'])
    lam_m1pp, t_m1 = m1_plus_plus(p, R)
    lam_p2, t_p2, fired = p2_plus_plus(p, R)
    lam_p3, c_val, indiff = p3_plus_plus(p, R)
    out = {}
    for name, lam, extra in [
        ('M1+',  lam_m1p,  {'scalar_gain_pT_R_dlam': float(p @ R @ (lam_m1p - p))}),
        ('C1++', lam_c1,   {'t_star': t_c1}),
        ('M1++', lam_m1pp, {'t_star': t_m1}),
        ('P2++', lam_p2,   {'t_star': t_p2, 'surgeries_fired': fired}),
        ('P3++', lam_p3,   {'conflict_energy': c_val, 'indifferent_C_is_zero': indiff}),
    ]:
        d = improvements(p, R, lam)
        returns_p = bool(np.allclose(lam, p, atol=1e-6))
        out[name] = dict(
            lam=lam.tolist(), returns_p=returns_p,
            min_improvement=float(d.min()),
            vector_safe=bool(np.all(d >= -1e-7)),
            l2_from_p=float(np.linalg.norm(lam - p)),
            certificate=('returns p: no direction improves ALL objectives over p in this '
                         'geometry, i.e. the floor is the singleton {p}. This is a theorem, '
                         'not a failure.') if (returns_p and name != 'M1+') else None,
            **extra)
    return out


def paired_rank_delta(heads_p, heads_l, p_vec):
    """Rank-normalised PAIRED Delta U_p: ranks pooled over BOTH models, per axis.

    Scale-free, non-circular (does not depend on B), and bootstrappable per prompt --
    unlike the set-level rank normalisation against B, which cannot be resampled
    over prompts.
    """
    n = heads_p.shape[0]
    dr = np.zeros(n)
    for j in range(heads_p.shape[1]):
        pooled = rankdata(np.r_[heads_p[:, j], heads_l[:, j]]) / (2 * n)
        dr += p_vec[j] * (pooled[n:] - pooled[:n])
    return dr


if CONFIG['RUN_FINAL_MERGE']:
    from datasets import load_dataset
    from transformers import AutoTokenizer

    assert R_COS_PATH.exists(), 'R_cos.npy missing; run the geometry phase.'
    assert REWARD_MATRIX_PATH.exists(), 'reward_matrix.npy missing; run reward collection.'
    R = np.load(R_COS_PATH)
    Reward = np.load(REWARD_MATRIX_PATH)
    vertex_rewards = Reward[:5]
    stl = np.diag(vertex_rewards).astype(float)          # r_k(theta_k) = STL reference
    assert np.all(np.abs(stl) > CONFIG['DM_DENOM_MIN']), \
        f'Delta m% denominator too small: {stl}. Refusing to divide.'

    adapter_paths = {a: RS_RUNS_DIR / f'ppo_{a}' / 'adapter' for a in CONFIG['ATTRIBUTES']}
    ds = load_dataset('nvidia/HelpSteer2', split=CONFIG['REWARD_PROMPT_SPLIT'])
    off = CONFIG['REWARD_PROMPT_OFFSET']
    eval_prompts = [str(ds[i]['prompt'])
                    for i in range(off, off + CONFIG['REWARD_NUM_PROMPTS'])]

    scorer = rs_ppo.ArmoRMHeadScorer(axis='helpfulness', model_id=CONFIG['ARMORM_MODEL'])
    scorer.validate_batching([(q, 'probe response') for q in eval_prompts[:8]])
    tok = AutoTokenizer.from_pretrained(str(SFT_MERGED))
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = 'left'

    def score_lambda(lmbda):
        """Real merge -> generate -> score. Returns the per-prompt head matrix [80, 5]."""
        rs_ppo.set_all_seeds(CONFIG['SEED'])           # identical rollouts across lambdas
        model = merge_theta(lmbda, adapter_paths, SFT_MERGED)
        responses = []
        for start in range(0, len(eval_prompts), 8):
            chunk = eval_prompts[start:start + 8]
            texts = [tok.apply_chat_template([{'role': 'user', 'content': q}],
                                             tokenize=False, add_generation_prompt=True)
                     for q in chunk]
            enc = tok(texts, return_tensors='pt', padding=True).to(model.device)
            n_new = random.randint(rs_ppo.CFG['output_min_len'], rs_ppo.CFG['output_max_len'])
            with torch.inference_mode():
                out = model.generate(**enc, max_new_tokens=n_new, do_sample=True,
                                     top_k=0, top_p=1.0, pad_token_id=tok.eos_token_id)
            responses.extend(tok.batch_decode(out[:, enc['input_ids'].shape[1]:],
                                              skip_special_tokens=True))
        heads = scorer.score_all_heads(eval_prompts, responses)
        del model; torch.cuda.empty_cache()
        return heads

    rng = np.random.default_rng(CONFIG['BOOTSTRAP_SEED'])
    results = []
    for name, p_list in PREFERENCES.items():
        p_vec = np.asarray(p_list, dtype=np.float64)
        pf = run_portfolio(p_vec, R, CONFIG)
        lam = np.asarray(pf['M1+']['lam'])             # M1+ is the primary mover

        heads_p = score_lambda(p_vec)                  # [80, 5]
        heads_l = score_lambda(lam)

        # --- PRIMARY (pre-registered): paired per-prompt Delta U_p, raw ArmoRM scale ---
        per_prompt = (heads_l - heads_p) @ p_vec       # [80]
        idx = rng.integers(0, len(per_prompt), (CONFIG['BOOTSTRAP_N'], len(per_prompt)))
        boots = per_prompt[idx].mean(axis=1)
        lo, hi = np.percentile(boots, [2.5, 97.5])

        # --- SECONDARY: paired RANK-normalised Delta U_p (same bootstrap indices) ---
        per_prompt_rank = paired_rank_delta(heads_p, heads_l, p_vec)
        boots_r = per_prompt_rank[idx].mean(axis=1)
        lo_r, hi_r = np.percentile(boots_r, [2.5, 97.5])

        # --- SECONDARY: level metrics (Delta m% vs the specialists) ---
        dm_p = float(np.mean((heads_p.mean(0) - stl) / stl) * 100)
        dm_l = float(np.mean((heads_l.mean(0) - stl) / stl) * 100)

        results.append({
            'preference': name,
            'regime': 'quality (negative control)' if name in QUALITY_PREFS
                      else ('complexity/verbosity' if name in CV_PREFS else 'uniform'),
            'p': p_vec.tolist(),
            'lambda_star_M1plus': lam.tolist(),
            'l2_lambda_minus_p': float(np.linalg.norm(lam - p_vec)),
            'delta_U_p_mean': float(per_prompt.mean()),
            'delta_U_p_ci95': [float(lo), float(hi)],
            'ci_excludes_zero': bool(lo > 0 or hi < 0),
            'delta_U_p_rank_mean': float(per_prompt_rank.mean()),
            'delta_U_p_rank_ci95': [float(lo_r), float(hi_r)],
            'rank_ci_excludes_zero': bool(lo_r > 0 or hi_r < 0),
            'delta_m_percent_at_p': dm_p,
            'delta_m_percent_at_lambda': dm_l,
            'delta_m_percent_gain': dm_l - dm_p,
            'portfolio': pf,
        })
        print(f"[{name:22s}] dU_p={per_prompt.mean():+.5f} CI=[{lo:+.5f},{hi:+.5f}] "
              f"excl0={str(bool(lo > 0 or hi < 0)):5s} "
              f"| rank={per_prompt_rank.mean():+.4f} | dm%gain={dm_l - dm_p:+.2f}")

    # --- aggregate: sign test over preferences (v5 analogue: 11/12, p=0.0032) ---
    deltas = [r['delta_U_p_mean'] for r in results]
    n_pos = int(sum(d > 0 for d in deltas))
    sign_p = float(binomtest(n_pos, len(deltas), 0.5, alternative='greater').pvalue)
    n_sig = int(sum(r['ci_excludes_zero'] and r['delta_U_p_mean'] > 0 for r in results))
    q_sig = int(sum(r['ci_excludes_zero'] and r['delta_U_p_mean'] > 0
                    for r in results if r['preference'] in QUALITY_PREFS))
    movers = {k: int(sum(not r['portfolio'][k]['returns_p'] for r in results))
              for k in ('M1+', 'C1++', 'M1++', 'P2++', 'P3++')}

    write_json(MERGE_RESULTS_PATH, {
        'circular_do_not_report_as_proxy_validation': True,
        'primary_endpoint': ('paired per-prompt Delta U_p (raw ArmoRM scale), bootstrap 95% CI '
                             '-- exactly the metric named in preregistration.json'),
        'rows': results,
        'n_preferences': len(results),
        'n_positive': n_pos,
        'sign_test_p_one_sided': sign_p,
        'n_significant_positive': n_sig,
        'n_quality_significant_positive': q_sig,
        'portfolio_movers': movers,
        'primary_success': bool(n_sig > 0),
        'upper_bound_verdict': (
            'f(p,R) beats p even on the quality axes in the Best-Case circular regime '
            '-> Wall A was NOT a hard cap; strong positive result.'
            if q_sig > 0 else
            'f(p,R) fails on the quality axes EVEN in the Best-Case circular regime '
            '-> UPPER-BOUND NEGATIVE RESULT: endpoint-linear coefficient correction cannot '
            'trace a nonlinear reward landscape, no matter how the specialists are trained. '
            'This is thesis-bearing and strictly stronger than the v5 finding.'),
    })
    df = pd.DataFrame(results)
    display(df[['preference', 'regime', 'delta_U_p_mean', 'delta_U_p_ci95',
                'ci_excludes_zero', 'delta_U_p_rank_mean', 'delta_m_percent_gain']])
    print(f"\nsign test: {n_pos}/{len(deltas)} positive, one-sided p = {sign_p:.4f}")
    print(f"portfolio movers (out of {len(results)} preferences): {movers}")
    del scorer; torch.cuda.empty_cache()
else:
    write_json(MERGE_RESULTS_PATH, {'pending': True,
                                    'reason': 'RUN_FINAL_MERGE is False; no fake merge numbers.'})
    print('RUN_FINAL_MERGE is False; primary endpoint not computed (no fake numbers).')


## Verdict, Summary und Zip

Das Verdict bleibt **STOP**, solange Pflichtartefakte fehlen. Zusaetzlich zum urspruenglichen Entwurf wird der **Plateau-Status** gefuehrt: plateaut eine Achse, ist die Upper-Bound-Lesart fuer sie ungueltig - das muss im Verdict stehen, nicht in einer Fussnote.


In [ ]:
VERDICT_PATH = OUTPUT_DIR / 'verdict.json'
SUMMARY_PATH = OUTPUT_DIR / 'summary.md'
ZIP_PATH = OUTPUT_DIR / CONFIG['OUTPUT_ZIP']

artifacts = {
    'preregistration': PREREGISTRATION_PATH,
    'head_sanity': HEAD_SANITY_PATH,
    'plateau_report': PLATEAU_PATH,
    'delta_norms': DELTA_NORMS_PATH,
    'R_gram': R_GRAM_PATH,
    'R_cos': R_COS_PATH,
    'geometry_precheck': GEOMETRY_PRECHECK_PATH,
    'lmc_check': LMC_CHECK_PATH,
    'reward_matrix': REWARD_MATRIX_PATH,
    'search_set_B': SEARCH_SET_PATH,
    'linearity_r2': LINEARITY_R2_PATH,
    'merge_results': MERGE_RESULTS_PATH,
}
loaded = {k: json.loads(Path(p).read_text(encoding='utf-8'))
          for k, p in artifacts.items() if Path(p).exists() and Path(p).suffix == '.json'}

reasons = []
missing = [k for k, p in artifacts.items() if not Path(p).exists()]
if missing:
    reasons.append('missing outputs: ' + ', '.join(missing))
if loaded.get('head_sanity', {}).get('passed') is not True:
    reasons.append('head sanity not passed')

plateau = loaded.get('plateau_report', {})
if plateau.get('pending'):
    reasons.append('plateau check pending')
elif plateau.get('_summary', {}).get('upper_bound_reading_valid') is False:
    reasons.append('reward plateaued on ' + ', '.join(plateau['_summary']['axes_plateaued'])
                   + ' -> upper-bound reading INVALID on those axes')

geom = loaded.get('geometry_precheck', {})
if geom.get('pending'):
    reasons.append('geometry pending')
elif geom.get('floor_collapsed') is True:
    reasons.append('floor collapsed (F_p = {p}) -> vector-safe methods return p; only M1+ moves')

lin = loaded.get('linearity_r2', {})
if lin.get('pending'):
    reasons.append('Wall-A / linearity R2 pending')

merge = loaded.get('merge_results', {})
if merge.get('pending'):
    reasons.append('primary endpoint (merge test) pending')

verdict = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'circularity_acknowledged': True,
    'retired_rqs': ['RQ2 proxy validity'],
    'valid_claims': ['upper bound', 'R-minus', 'wall A (R2)', 'LMC'],
    'invalid_claims': ['proxy validity', 'generalization beyond this reward model'],
    'circular_do_not_report_as_proxy_validation': True,
    'upper_bound_reading_valid': plateau.get('_summary', {}).get('upper_bound_reading_valid'),
    'floor_collapsed': geom.get('floor_collapsed'),
    'wall_a_stands': lin.get('wall_a_stands'),
    'primary_success': merge.get('primary_success'),
    'upper_bound_verdict': merge.get('upper_bound_verdict'),
    'decision': 'GO' if not reasons else 'STOP',
    'reasons': reasons,
    'config': CONFIG,
}
write_json(VERDICT_PATH, verdict)

summary = ['# 08 - RS-PPO gegen ArmoRM (bewusst zirkulaer)', '']
summary += ['## Was dieses Notebook zeigen kann und was nicht', '',
            '**Gueltig:** obere Schranke (Best-Case-Regime), R-minus, Wall A / R2, LMC.',
            '',
            '**Ungueltig:** RQ2 Proxy-Validitaet und jede Generalisierung ueber ArmoRM hinaus.',
            '',
            'PPO wird bewusst gegen ArmoRM trainiert und gegen ArmoRM evaluiert. Das ist zirkulaer '
            'und in `preregistration.json` explizit anerkannt. Spearman-/Gate-Zahlen aus diesem '
            'Lauf duerfen **nicht** als Proxy-Validierung berichtet werden.', '',
            '**Bedingung fuer die obere Schranke:** kurzer Horizont. Plateaut die Reward-Kurve, '
            'ist `delta ~ eta*grad r` verletzt und die Best-Case-Lesart faellt.', '',
            f"## Verdict: **{verdict['decision']}**", '']
summary += ['Reasons:'] + (['- ' + r for r in reasons] if reasons
                           else ['- all preregistered criteria satisfied'])
summary += ['', '## Kernbefunde', '',
            f"- upper_bound_reading_valid: {verdict['upper_bound_reading_valid']}",
            f"- floor_collapsed: {verdict['floor_collapsed']}",
            f"- wall_a_stands: {verdict['wall_a_stands']}",
            f"- primary_success (Delta U_p > 0, CI excl. 0): {verdict['primary_success']}",
            '']
if verdict.get('upper_bound_verdict'):
    summary += ['> ' + verdict['upper_bound_verdict'], '']
summary += ['## Outputs', ''] + [f'- {k}: {Path(p).name}' for k, p in artifacts.items()]
SUMMARY_PATH.write_text('\n'.join(summary) + '\n', encoding='utf-8')

with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for p in [*artifacts.values(), VERDICT_PATH, SUMMARY_PATH]:
        p = Path(p)
        if p.exists():
            archive.write(p, arcname=p.name)
            print('Added:', p.name)

print('\nOutput zip:', ZIP_PATH)
print('Verdict   :', verdict['decision'])
for r in reasons:
    print('  -', r)

try:
    from google.colab import files
    files.download(str(ZIP_PATH))
except Exception as error:
    print('Download manually from:', ZIP_PATH, '|', repr(error))
